# Visualize Guided Pair Dataset

This notebook visually checks the dataset wrapper added for similarity-guided mixing. It does **not** apply SimMixUp or SimCutMix yet. It only shows which full-subset neighbor `j` is selected for an anchor sample `i`.

Each row shows:

- left: anchor image `x_i`, label `y_i`, original CIFAR training index `idx_i`
- right: selected neighbor image `x_j`, label `y_j`, original CIFAR training index `idx_j`

For `class_aware`, labels should match. For `class_agnostic`, labels may differ.

In [ ]:
from pathlib import Path

# Change these values to inspect a different split or neighbor mode.
DATASET = "cifar100"          # "cifar10" or "cifar100"
K = 100
SUBSET_SEED = 0
MODE = "class_agnostic"         # "class_aware" or "class_agnostic"
PAIR_SAMPLING = "weighted"    # "uniform" or "weighted"
PAIR_SEED = 0
NUM_PAIRS = 12
VISUAL_SAMPLE_SEED = 7

# Keep False for offline/cluster-style checks. Set True only if CIFAR is missing locally.
DOWNLOAD_DATA = False

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

DATA_ROOT = ROOT / "data" / "raw"
SPLIT_ROOT = ROOT / "data" / "splits"
NEIGHBOR_ROOT = ROOT / "results" / "experiments" / "shared" / "neighbors"

ROOT

In [ ]:
import json
import sys
from collections import Counter

import matplotlib.pyplot as plt
import torch
from torchvision import transforms
from torchvision.datasets import CIFAR10, CIFAR100

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from src.data.indexed_dataset import GuidedPairDataset


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def resolve_repo_path(path_text):
    path = Path(path_text)
    return path if path.is_absolute() else ROOT / path


def find_neighbor_path(dataset, k, subset_seed, mode):
    neighbor_dir = NEIGHBOR_ROOT / dataset / f"k{k}_seed{subset_seed}"
    metadata_path = neighbor_dir / "metadata.json"
    if metadata_path.exists():
        metadata = load_json(metadata_path)
        if mode in metadata.get("neighbors", {}):
            return resolve_repo_path(metadata["neighbors"][mode]["path"])

    matches = sorted(neighbor_dir.glob(f"neighbors_{mode}_K*.pt"))
    if not matches:
        raise FileNotFoundError(
            f"No neighbor file found for {dataset}, k={k}, seed={subset_seed}, mode={mode}. "
            f"Expected metadata or neighbors_{mode}_K*.pt under {neighbor_dir}"
        )
    return matches[-1]


def build_pair_dataset(mode=MODE, pair_sampling=PAIR_SAMPLING, pair_seed=PAIR_SEED):
    split_path = SPLIT_ROOT / DATASET / f"k{K}_seed{SUBSET_SEED}.json"
    split_info = load_json(split_path)
    train_indices = split_info["train_indices"]

    dataset_cls = CIFAR10 if DATASET == "cifar10" else CIFAR100
    base_dataset = dataset_cls(
        root=DATA_ROOT,
        train=True,
        download=DOWNLOAD_DATA,
        transform=transforms.ToTensor(),
    )

    neighbor_path = find_neighbor_path(DATASET, K, SUBSET_SEED, mode)
    neighbor_payload = torch.load(neighbor_path, map_location="cpu")
    pair_dataset = GuidedPairDataset(
        base_dataset=base_dataset,
        train_indices=train_indices,
        neighbor_index=neighbor_payload,
        pair_sampling=pair_sampling,
        mode=mode,
        seed=pair_seed,
    )
    return pair_dataset, base_dataset, neighbor_payload, neighbor_path


pair_dataset, base_dataset, neighbor_payload, neighbor_path = build_pair_dataset()

print(f"Loaded {DATASET} k={K} seed={SUBSET_SEED}")
print(f"mode: {MODE}")
print(f"pair_sampling: {PAIR_SAMPLING}")
print(f"num_train: {len(pair_dataset)}")
print(f"neighbor file: {neighbor_path}")
print(f"neighbors per anchor: {neighbor_payload['neighbor_indices'].shape[1]}")

In [ ]:
def image_for_plot(image):
    return image.permute(1, 2, 0).detach().cpu().clamp(0, 1)


def label_name(label):
    return base_dataset.classes[int(label)]


generator = torch.Generator().manual_seed(VISUAL_SAMPLE_SEED)
positions = torch.randperm(len(pair_dataset), generator=generator)[:NUM_PAIRS].tolist()

fig, axes = plt.subplots(len(positions), 2, figsize=(7, 2.3 * len(positions)))
if len(positions) == 1:
    axes = axes.reshape(1, 2)

for row, position in enumerate(positions):
    image_i, label_i, image_j, label_j, idx_i, idx_j = pair_dataset[position]
    same_label = int(label_i) == int(label_j)
    title_color = "darkgreen" if same_label else "darkred"

    axes[row, 0].imshow(image_for_plot(image_i))
    axes[row, 0].set_title(
        f"anchor idx={idx_i}\ny={int(label_i)} ({label_name(label_i)})",
        fontsize=9,
    )
    axes[row, 0].axis("off")

    axes[row, 1].imshow(image_for_plot(image_j))
    axes[row, 1].set_title(
        f"neighbor idx={idx_j}\ny={int(label_j)} ({label_name(label_j)})",
        fontsize=9,
        color=title_color,
    )
    axes[row, 1].axis("off")

fig.suptitle(
    f"GuidedPairDataset: {DATASET} k={K} seed={SUBSET_SEED} mode={MODE} sampling={PAIR_SAMPLING}",
    fontsize=12,
)
plt.tight_layout()
plt.show()

## Quick sanity counts

This checks how many sampled pairs have the same label versus different labels for the active mode. In `class_aware`, `different_label` should be `0`.

In [ ]:
def pair_label_counts(dataset):
    counts = Counter()
    for position in range(len(dataset)):
        _, label_i, _, label_j, _, _ = dataset[position]
        key = "same_label" if int(label_i) == int(label_j) else "different_label"
        counts[key] += 1
    return counts


pair_label_counts(pair_dataset)

## Optional: compare both modes

Run this after both neighbor files exist. It is a compact check that `class_aware` stays same-class while `class_agnostic` can select cross-class partners.

In [ ]:
for mode in ["class_aware", "class_agnostic"]:
    try:
        dataset_for_mode, _, _, path_for_mode = build_pair_dataset(mode=mode)
        print(mode, path_for_mode.name, pair_label_counts(dataset_for_mode))
    except FileNotFoundError as exc:
        print(mode, exc)